# NESTML/SpiNNaker single neuron characterisation

In this notebook, we will instantiate the newly created neuron and check the postsynaptic responses to incoming spikes.

To do this, we create a simple network consisting of a spike generator (a device that emits spikes at predefined times), connected to a NESTML iaf_psc_exp neuron (the "spiking neuron"), which itself spikes and sends the spikes to a second iaf_psc_exp neuron (the "receiving neuron"). This allows us to see the emission and handling of spikes in the new neuron model.

In [1]:
%matplotlib inline

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs

import pyNN.spiNNaker as pynn
from pyNN.utility.plotting import Figure, Panel
from spynnaker.pyNN.data import SpynnakerDataView
SpynnakerDataView.register_binary_search_path("/opt/app-root/src")

from iaf_psc_exp_neuron_nestml import iaf_psc_exp_neuron_nestml

## Creating and simulating the network

In [2]:
exc_input = "exc_spikes"
inh_input = "inh_spikes"
membranePot = "V_m"
synapticRsp = "I_syn_exc"

# Set the run time of the execution
run_time = 200

# Set the time step of the simulation in milliseconds
time_step = 1

# Set the i_offset current
i_offset = 0.0

# Set the weight of input spikes
weight = 2000

# Set the times at which to input a spike
spike_times = [1, 5, 100]
spike_times_inh = [50]

pynn.setup(time_step)

spikeArray = {"spike_times": spike_times}
excitation = pynn.Population(1, pynn.SpikeSourceArray(**spikeArray), label="input")

spikeArray_inh = {"spike_times": spike_times_inh}
inhibition = pynn.Population(1, pynn.SpikeSourceArray(**spikeArray_inh), label="input")

spiking_neuron = pynn.Population(1, iaf_psc_exp_neuron_nestml(), label="iaf_psc_exp_neuron_nestml_spiking")
pynn.Projection(
    excitation, spiking_neuron,
    pynn.OneToOneConnector(), receptor_type=exc_input,
    synapse_type=pynn.StaticSynapse(weight=weight))
pynn.Projection(
    inhibition, spiking_neuron,
    pynn.OneToOneConnector(), receptor_type=inh_input,
    synapse_type=pynn.StaticSynapse(weight=weight))

receiving_neuron = pynn.Population(1, iaf_psc_exp_neuron_nestml(), label="iaf_psc_exp_neuron_nestml_receiving")
pynn.Projection(
    spiking_neuron, receiving_neuron,
    pynn.OneToOneConnector(), receptor_type=exc_input,
    synapse_type=pynn.StaticSynapse(weight=weight))

spiking_neuron.record(["spikes"])
spiking_neuron.record([membranePot])
spiking_neuron.record([synapticRsp])

receiving_neuron.record(["spikes"])
receiving_neuron.record([membranePot])
receiving_neuron.record([synapticRsp])

# run the simulation
pynn.run(run_time)

# grab recorded data
spikes_spiking_neuron = spiking_neuron.get_data("spikes")
v_spiking_neuron = spiking_neuron.get_data(membranePot)
i_syn_exc_spiking_neuron = spiking_neuron.get_data(synapticRsp)

spikes_receiving_neuron = receiving_neuron.get_data("spikes")
v_receiving_neuron = receiving_neuron.get_data(membranePot)
i_syn_exc_receiving_neuron = receiving_neuron.get_data(synapticRsp)

pynn.end()

2026-06-11 13:53:50 INFO: Read configs files: /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_utilities/spinn_utilities.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_machine/spinn_machine.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/pacman/pacman.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinnman/spinnman.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_front_end_common/interface/spinnaker.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/

Plot the recorded data from the simulation:

In [3]:
combined_spikes = spikes_spiking_neuron.segments[0].spiketrains
for spike in spikes_receiving_neuron.segments[0].spiketrains:
    combined_spikes.append(spike)

Figure(
    Panel(combined_spikes,
          xlabel="Time (ms)",
          data_labels=["spikes"],
          yticks=True, xlim=(0, run_time), xticks=True),

    Panel(i_syn_exc_spiking_neuron.segments[0].filter(name=synapticRsp)[0],
          xlabel="Time (ms)",
          ylabel="Synaptic response",
          data_labels=[spiking_neuron.label],
          yticks=True, xlim=(0, run_time), xticks=True),

    Panel(v_spiking_neuron.segments[0].filter(name=membranePot)[0],
          xlabel="Time (ms)",
          ylabel="Membrane potential (mV)",
          data_labels=[spiking_neuron.label],
          yticks=True, xlim=(0, run_time), xticks=True),

    Panel(i_syn_exc_receiving_neuron.segments[0].filter(name=synapticRsp)[0],
          xlabel="Time (ms)",
          ylabel="Synaptic response",
          data_labels=[receiving_neuron.label],
          yticks=True, xlim=(0, run_time), xticks=True),

    Panel(v_receiving_neuron.segments[0].filter(name=membranePot)[0],
          xlabel="Time (ms)",
          ylabel="Membrane potential (mV)",
          data_labels=[receiving_neuron.label],
          yticks=True, xlim=(0, run_time), xticks=True),

    title="Generated: Two chained neurons",
    annotations="Simulated with {}".format(pynn.name())
)
plt.savefig("spinnaker_iaf_psc_exp_single_neuron.png")

## Acknowledgements

The authors would like to thank Andrew Rowley for his kind and helpful discussion and feedback.

## Copyright

This file is part of NEST.

Copyright (C) 2004 The NEST Initiative

NEST is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 2 of the License, or (at your option) any later version.

NEST is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with NEST. If not, see http://www.gnu.org/licenses/.